# Phase 1 — SimCLR reference encoder (Kaggle)

Trains the Phase-1 SimCLR reference encoder (2 seeds) + the supervised-reference ceiling on Shapes3D, then runs the encoder-quality gate (SimCLR vs random-encoder vs supervised).

**Two ways to run:**
- **From scratch** — run every section top to bottom (~10 h/seed for the two SimCLR runs).
- **Fast path (reuse encoders you already trained)** — upload the trained `backbone.pt` files in **section 4c**, then skip the matching training sections and jump to the gate. Recommended: reuse the two expensive SimCLR seeds and just retrain the cheap supervised encoder (section 7, ~15 min).

**Before running** (right sidebar → Settings):
- **Accelerator → GPU T4 x2** (not P100: the P100 is CUDA capability sm_60, which Kaggle's current PyTorch no longer supports — you'd get a 'no kernel image is available' error. The T4 is sm_75 and works. The code is single-GPU, so only one of the two T4s is used; that's expected.)
- **Internet → On** (needed to clone the repo, `pip install`, and download the dataset).

Everything lives on Kaggle's local SSD (`/kaggle/working`), which is fast and persists for the session. To keep checkpoints across sessions, **Save Version** (commit) — outputs in `/kaggle/working` are stored with the version. Training checkpoints every epoch and auto-resumes from `last_ckpt.pt`: if a cell stops partway, just re-run it.

In [ ]:
!nvidia-smi

## 1. Clone the repo

Cloned onto local SSD (`/kaggle/working`). Run cells below `cd` in by absolute path so they still work after a kernel restart (which clears the `%cd` but leaves files on disk).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 2. Install dependencies

Install the package without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Settings > Accelerator > GPU")

## 3. Download Shapes3D + build the image cache

`--download` fetches the ~255 MB HDF5 to local disk; `--build-cache` decompresses it once into an uncompressed memory-mapped array (`data/raw/3dshapes.images.npy`, ~5.9 GB on local SSD). Training then mmaps that file, so images page in on demand and RAM stays flat regardless of `subset` — this is what fixes the out-of-memory crashes. Both steps are idempotent and cached across the session.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 4. Calibrate runtime (optional, one epoch)

Run one epoch and note the wall time; ×100 ≈ the full-run estimate. This writes the epoch-0 checkpoint, so the full run below resumes from epoch 1 — no wasted compute. RAM should stay low and flat (mmap). On a T4 expect roughly **1–1.5 s/it** (~6 min/epoch, so ~10 h for the full 100-epoch run per seed). The first iteration is slower while workers spin up — judge from the steady-state rate.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml \
    --set run.seed=0 run.epochs=1 run.num_workers=2

## 4b. Resume from a downloaded checkpoint (optional)

Only run this if you're picking up a run after a session ended and you have a
previously downloaded `last_ckpt.pt` on your machine. **Skip the section 4
calibration cell above if you're resuming** — it forces `run.epochs=1`, and if
the restored checkpoint is already past epoch 1 the script's epoch loop runs
zero times and errors out.

Steps:
1. Right sidebar → **Add Input → Upload → New Dataset**, upload your
   downloaded `last_ckpt.pt` file(s) (folder structure doesn't matter, the
   cell below finds them by filename), create the dataset, and attach it to
   this notebook.
2. Set `RUN_IDS` below to the run directories these checkpoints belong to
   (e.g. `standard_strong_seed0`, `standard_strong_seed1`, `supervised_seed0`)
   — match them up by the order you uploaded, or just rerun this cell after
   adjusting the mapping if it guesses wrong.
3. Run the cell — it copies each `last_ckpt.pt` found under `/kaggle/input`
   into `results/encoders/<run_id>/last_ckpt.pt`. Training picks it up
   automatically (`run.resume: true` is the default) and prints `resumed
   from last_ckpt.pt at epoch N` when you run the real training cell.

In [ ]:
import shutil
from pathlib import Path

# Map each uploaded last_ckpt.pt to the run directory it belongs to, in the
# order they're found under /kaggle/input. Adjust to match what you uploaded.
RUN_IDS = ["standard_strong_seed0"]

found = sorted(Path("/kaggle/input").rglob("last_ckpt.pt"))
print(f"found {len(found)} checkpoint file(s):")
for f in found:
    print(" ", f)

if len(found) != len(RUN_IDS):
    print(
        f"\nWARNING: found {len(found)} checkpoint(s) but RUN_IDS has "
        f"{len(RUN_IDS)} entries — fix RUN_IDS to match, then rerun this cell."
    )
else:
    for src, run_id in zip(found, RUN_IDS):
        dest_dir = Path("results/encoders") / run_id
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / "last_ckpt.pt"
        shutil.copy(src, dest)
        print(f"restored {src} -> {dest}")

## 4c. (Fast path) Reuse encoders you already trained

If you already trained some encoders (e.g. on Colab) and don't want to spend ~10 h/seed retraining, upload their final **`backbone.pt`** files and restore them here, then skip the matching training section.

**Recommended:** upload the two expensive SimCLR backbones (`standard_strong_seed0`, `standard_strong_seed1`) and **retrain only the supervised encoder** in section 7 — it's ~15 min, cheaper than shuttling the file around.

Upload steps: right sidebar → **Add Input → Upload → New Dataset**. Either keep the folder names (`.../standard_strong_seed0/backbone.pt`) or rename each file to include its run id (`standard_strong_seed0.pt`). The cell below matches by run id **in the path**, so upload order doesn't matter. Attach the dataset, then run the cell.

The two SimCLR `backbone.pt` files are 44 MB each. The supervised one you can either upload too (add `"supervised_seed0"` to `RUN_IDS`) or just retrain in section 7.

In [ ]:
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")

# Final backbones to restore from /kaggle/input (skip retraining these).
# Supervised is cheap — leave it out and train it in section 7, or add it here.
RUN_IDS = ["standard_strong_seed0", "standard_strong_seed1"]

for run_id in RUN_IDS:
    hits = [p for p in Path("/kaggle/input").rglob("*.pt")
            if run_id in p.as_posix() and (p.name == "backbone.pt" or run_id in p.name)]
    if not hits:
        print(f"— {run_id}: no backbone.pt found under /kaggle/input — train it below")
        continue
    dst = REPO / "results" / "encoders" / run_id / "backbone.pt"
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(hits[0], dst)
    print(f"✓ {run_id}: restored from {hits[0]} ({dst.stat().st_size // 1024 // 1024} MB)")

print("\nRestored backbones let you SKIP their training sections. "
      "Then run section 7 (supervised) if not restored, and section 8 (the gate).")

## 5. Train the reference encoder — seed 0

Checkpoints every epoch (~10 h total on a T4). A Kaggle session caps around 12 h, so this fits in one session; if it stops, **re-run this cell** to resume from the last saved epoch. **Save Version** periodically so `/kaggle/working` checkpoints persist if the session ends.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml \
    --set run.seed=0 run.num_workers=2

## 6. Train the reference encoder — seed 1 (seed-to-seed reproducibility check)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml \
    --set run.seed=1 run.num_workers=2

## 7. Train the supervised-reference encoder (recoverability ceiling)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.supervised \
    --config configs/run/supervised.yaml \
    --set run.seed=0 run.num_workers=2

## 8. Run the encoder-quality gate

Per-factor normalized recoverability for SimCLR vs random-encoder vs supervised-reference → PASS/FAIL against the Phase-1 thresholds.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.eval.quality_gate \
    --config configs/run/reference_cuda.yaml \
    --simclr results/encoders/standard_strong_seed0/backbone.pt \
             results/encoders/standard_strong_seed1/backbone.pt \
    --supervised results/encoders/supervised_seed0/backbone.pt \
    --random-seed 0 \
    --out results/quality_gate/reference.json

## 9. Print the gate report

In [ ]:
import json
print(json.dumps(json.load(open("results/quality_gate/reference.json")), indent=2))

## 10. Phase 2 — validate the probe instrument on controls

Runs the probe-capacity ladder (linear · 64 · 256 · 256→128) and the G / S / ε_G
metric layer, then checks the instrument on the controls (the Phase-2 gate):
1. probe params strictly increase across the ladder,
2. the random-LABEL control collapses on held-out test (S wiring),
3. the random-vs-random-encoder ε_G null is centered ~0,
4. the trained encoder reads shape well above its own random-label floor.

Reuses the two SimCLR `backbone.pt` restored in section 4c — **no retraining**.
Pull the latest code first (adds `src/probes/` + `configs/probe/ladder.yaml`).

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && git pull

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.validate \
    --config configs/probe/ladder.yaml \
    --simclr /kaggle/input/datasets/chinesegorilla99/backbones/standard_strong_seed0.pt \
             /kaggle/input/datasets/chinesegorilla99/backbones/standard_strong_seed1.pt \
    --random-seed 0 1 2 \
    --subsample 8000 --epochs 60 --num-workers 2 \
    --out results/probes/phase2_validation.json

## 11. Print the Phase-2 validation report

In [ ]:
import json
r = json.load(open("results/probes/phase2_validation.json"))
print("VERDICT:", "PASS" if r["checks"]["passed"] else "FAIL")
print(json.dumps(r["checks"], indent=2))
print("\nepsilon source:", r["report"]["epsilon_source"], "| flips_primary:", r["report"]["flips_primary"])
for fac, rows in r["report"]["table"].items():
    print(fac, [(c["rung"], round(c["G"], 3), round(c["S"], 3), c["case"]) for c in rows])